# 02 · Train SignNet (CNN tự thiết kế) trên crop 64x64

GPU T4 (~3-4 phút/cấu hình, mỗi cấu hình bị chặn cứng 4.5 phút -> cả notebook < 15 phút).
Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Input: code + output notebook 01 (`crops.tgz`). Kaggle: add output của notebook 01 làm input;
Colab: chạy notebook 01 trước, file nằm sẵn trên Drive.
So 3 cách xử lý lệch lớp: CE thường / + sampler căn bậc 2 / + sampler + loss có trọng số.
Thước đo chính là **macro-F1** (mỗi lớp nặng như nhau), accuracy bị lớp đông áp đảo.

In [ ]:
import glob, json, os, sys

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/dashcam/__init__.py", recursive=True)[0].rsplit("/dashcam", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vn-dashcam-vision" if IN_COLAB else os.path.abspath("vn-dashcam-vision")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vn-dashcam-vision {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vn-dashcam-vision" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in (["/kaggle/input"] if IN_KAGGLE else []) + [WORK, REPO]:
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


# Colab: đọc hàng nghìn ảnh từ Google Drive rất chậm -> dataset để ở ổ cục bộ /content (mất khi ngắt,
# chạy lại cell là có), còn model / kết quả / checkpoint YOLO thì nằm trong WORK trên Drive.
DATA = "/content/data" if IN_COLAB else "datasets"
DS = f"{DATA}/vnts"
os.makedirs(DATA, exist_ok=True)

TGZ = find("crops.tgz", "chạy notebook 01 trước")
if not os.path.exists(f"{DS}/crops"):
    !tar xzf {TGZ} -C {DATA}
!pip install -q onnx onnxruntime

In [ ]:
runs = {
    "ce": "--sampler none",
    "sqrt": "--sampler sqrt",
    "sqrt+weighted": "--sampler sqrt --weighted-loss",
}
for name, flags in runs.items():
    if os.path.exists(f"runs/cls_{name}/log.json"):  # log.json chỉ ghi khi train xong
        print(name, "đã train xong, bỏ qua")
        continue
    !python -m dashcam.train_classifier --crops {DS}/crops --names {DS}/names.json \
        --out runs/cls_{name} --epochs 40 --max-min 4.5 {flags}

In [ ]:
import numpy as np, torch
from dashcam.classifier import SignNet
from dashcam.train_classifier import metrics, predict

names = json.load(open(f"{DS}/names.json"))
stats = json.load(open(f"{DS}/class_stats.json"))["train"]
te = np.load(f"{DS}/crops/test.npz")
rare = {i for i, n in enumerate(names) if stats.get(n, 0) < 50}
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
res = {}
for name in runs:
    st = torch.load(f"runs/cls_{name}/best.pt", map_location=dev)
    m = SignNet(len(names), st["width"]).to(dev); m.load_state_dict(st["model"])
    p = predict(m, te["x"], dev)
    mt = metrics(p, te["y"], len(names))
    rare_rec = np.mean([r for c, r in mt["per_class_recall"].items() if c in rare]) if rare else None
    res[name] = {"acc": round(mt["acc"], 4), "macro_f1": round(mt["macro_f1"], 4),
                 "recall_lớp_hiếm": None if rare_rec is None else round(float(rare_rec), 4)}
import pandas as pd
pd.DataFrame(res).T

## Nhầm lẫn hay gặp (cấu hình tốt nhất)

In [ ]:
best = max(res, key=lambda k: res[k]["macro_f1"])
st = torch.load(f"runs/cls_{best}/best.pt", map_location=dev)
m = SignNet(len(names), st["width"]).to(dev); m.load_state_dict(st["model"])
pred = predict(m, te["x"], dev).argmax(1)
from collections import Counter
conf = Counter((names[g], names[p]) for g, p in zip(te["y"], pred) if g != p)
pd.DataFrame([(g, p, n) for (g, p), n in conf.most_common(15)], columns=["đúng", "đoán", "số lần"])

In [ ]:
!mkdir -p models && cp runs/cls_{best}/best.pt runs/signnet_best.pt
!python -m dashcam.export --signnet runs/signnet_best.pt --calib-crops {DS}/crops/train.npz --out models
json.dump(res, open("models/classifier_results.json", "w"), ensure_ascii=False, indent=1)